In [1]:
from pathlib import Path
from preprocessing import Preprocesar

csv_path = Path(r"data\raw\compas-scores-two-years.csv")

# Cambiar por "CorrelationRemover", "PrototypeRepresentationLearner" o None:
compas_preprocessed, compas_audit_context = Preprocesar(csv_path, preprocessing_method="PrototypeRepresentationLearner")

display(compas_preprocessed.head())
display(compas_audit_context.head())

,prototype_0,prototype_1,two_year_recid
0,0.222960,0.777040,0
1,0.674250,0.325750,1
2,0.832128,0.167872,1
3,0.492094,0.507906,0
4,0.473768,0.526232,1


,entity_id,race,sex,age_cat,label_value
0,1,Other,Male,Greater than 45,0
1,3,African-American,Male,25 - 45,1
2,4,African-American,Male,Less than 25,1
3,7,Other,Male,25 - 45,0
4,8,Caucasian,Male,25 - 45,1


In [2]:
from train_model import train_model

result, audit_predictions = train_model(compas_preprocessed, compas_audit_context)

display(audit_predictions)

,entity_id,model_id,feature_set,fold,race,sex,age_cat,label_value,score
0,3,LR_8_features,8,1,African-American,Male,25 - 45,1,0
1,14,LR_8_features,8,1,Caucasian,Male,25 - 45,0,1
2,42,LR_8_features,8,1,African-American,Male,25 - 45,1,0
3,45,LR_8_features,8,1,Caucasian,Male,Greater than 45,0,0
4,70,LR_8_features,8,1,African-American,Male,25 - 45,0,1
...,...,...,...,...,...,...,...,...,...
6167,10957,LR_8_features,8,10,African-American,Male,Greater than 45,1,0
6168,10972,LR_8_features,8,10,Caucasian,Male,25 - 45,0,0
6169,10977,LR_8_features,8,10,African-American,Male,25 - 45,1,0
6170,10989,LR_8_features,8,10,African-American,Male,25 - 45,0,0


In [3]:
from auditar import auditar_modelo
# ignore warnings
import warnings
warnings.filterwarnings("ignore")

resultado = auditar_modelo(audit_predictions)

In [4]:
ABSOLUTE_COLUMNS = [
    "model_id", "attribute_name", "attribute_value", "group_size", "prev", "pprev",
    "tpr", "tnr", "fpr", "fnr", "fdr", "for", "precision", "npv",
]
DISPARITY_COLUMNS = [
    "model_id", "attribute_name", "attribute_value", "group_size",
    "fpr_disparity", "fnr_disparity", "pprev_disparity", "fdr_disparity",
    "precision_disparity", "fpr_ref_group_value", "fnr_ref_group_value",
]
FAIRNESS_COLUMNS = [
    "model_id", "attribute_name", "attribute_value", "group_size",
    "FPR Parity", "FNR Parity", "Statistical Parity", "Impact Parity",
    "Equalized Odds", "TypeI Parity", "TypeII Parity",
    "Unsupervised Fairness", "Supervised Fairness",
]

resultado["group_metrics"] = resultado["group_metrics"][ABSOLUTE_COLUMNS]
resultado["disparities"] = resultado["disparities"][DISPARITY_COLUMNS]
resultado["group_fairness"] = resultado["group_fairness"][FAIRNESS_COLUMNS]

In [5]:
print("Metricas de grupo:")
display(resultado["group_metrics"])
print("\nDisparidades respecto al grupo de referencia:")
display(resultado["disparities"])
print("\nEquidad de grupo:")
display(resultado["group_fairness"])

Metricas de grupo:


,model_id,attribute_name,attribute_value,group_size,prev,pprev,tpr,tnr,fpr,fnr,fdr,for,precision,npv
0,LR_8_features,race,African-American,3175,0.523150,0.490394,0.544852,0.569353,0.430647,0.455148,0.418754,0.467244,0.581246,0.532756
1,LR_8_features,race,Asian,31,0.258065,0.322581,0.375000,0.695652,0.304348,0.625000,0.700000,0.238095,0.300000,0.761905
2,LR_8_features,race,Caucasian,2103,0.390870,0.391821,0.464720,0.654957,0.345043,0.535280,0.536408,0.344019,0.463592,0.655981
3,LR_8_features,race,Hispanic,509,0.371316,0.345776,0.428571,0.703125,0.296875,0.571429,0.539773,0.324324,0.460227,0.675676
4,LR_8_features,race,Native American,11,0.454545,0.181818,0.400000,1.000000,0.000000,0.600000,0.000000,0.333333,1.000000,0.666667
5,LR_8_features,race,Other,343,0.361516,0.067055,0.145161,0.977169,0.022831,0.854839,0.217391,0.331250,0.782609,0.668750
6,LR_8_features,sex,Female,1175,0.351489,0.406809,0.457627,0.620735,0.379265,0.542373,0.604603,0.321377,0.395397,0.678623
7,LR_8_features,sex,Male,4997,0.479488,0.423054,0.501669,0.649366,0.350634,0.498331,0.431410,0.414152,0.568590,0.585848
8,LR_8_features,age_cat,25 - 45,3532,0.464609,0.367214,0.400366,0.661555,0.338445,0.599634,0.493446,0.440268,0.506554,0.559732
9,LR_8_features,age_cat,Greater than 45,1293,0.320186,0.000000,0.000000,1.000000,0.000000,1.000000,NaN,0.320186,NaN,0.679814



Disparidades respecto al grupo de referencia:


,model_id,attribute_name,attribute_value,group_size,fpr_disparity,fnr_disparity,pprev_disparity,fdr_disparity,precision_disparity,fpr_ref_group_value,fnr_ref_group_value
0,LR_8_features,race,African-American,3175,1.248098,0.850298,1.251575,0.780664,1.253787,Caucasian,Caucasian
1,LR_8_features,race,Asian,31,0.882058,1.167614,0.823285,1.304977,0.647120,Caucasian,Caucasian
2,LR_8_features,race,Caucasian,2103,1.000000,1.000000,1.000000,1.000000,1.000000,Caucasian,Caucasian
3,LR_8_features,race,Hispanic,509,0.860400,1.067532,0.882484,1.006273,0.992742,Caucasian,Caucasian
4,LR_8_features,race,Native American,11,0.000000,1.120909,0.464034,0.000000,2.157068,Caucasian,Caucasian
5,LR_8_features,race,Other,343,0.066169,1.596994,0.171138,0.405272,1.688140,Caucasian,Caucasian
6,LR_8_features,sex,Female,1175,1.081654,1.088380,0.961600,1.401458,0.695400,Male,Male
7,LR_8_features,sex,Male,4997,1.000000,1.000000,1.000000,1.000000,1.000000,Male,Male
8,LR_8_features,age_cat,25 - 45,3532,1.000000,1.000000,1.000000,1.000000,1.000000,25 - 45,25 - 45
9,LR_8_features,age_cat,Greater than 45,1293,0.000000,1.667683,0.000000,NaN,NaN,25 - 45,25 - 45



Equidad de grupo:


,model_id,attribute_name,attribute_value,group_size,FPR Parity,FNR Parity,Statistical Parity,Impact Parity,Equalized Odds,TypeI Parity,TypeII Parity,Unsupervised Fairness,Supervised Fairness
0,LR_8_features,race,African-American,3175,True,True,False,False,True,False,False,False,False
1,LR_8_features,race,Asian,31,True,True,False,True,True,False,False,False,False
2,LR_8_features,race,Caucasian,2103,True,True,True,True,True,True,True,True,True
3,LR_8_features,race,Hispanic,509,True,True,False,True,True,True,True,False,True
4,LR_8_features,race,Native American,11,False,True,False,False,False,False,True,False,False
5,LR_8_features,race,Other,343,False,False,False,False,False,False,False,False,False
6,LR_8_features,sex,Female,1175,True,True,False,True,True,False,False,False,False
7,LR_8_features,sex,Male,4997,True,True,True,True,True,True,True,True,True
8,LR_8_features,age_cat,25 - 45,3532,True,True,True,True,True,True,True,True,True
9,LR_8_features,age_cat,Greater than 45,1293,False,False,False,False,False,False,False,False,False


In [6]:
resultado['attribute_fairness']

,model_id,score_threshold,attribute_name,Statistical Parity,Impact Parity,FDR Parity,FPR Parity,FOR Parity,FNR Parity,TPR Parity,TNR Parity,NPV Parity,Precision Parity,TypeI Parity,TypeII Parity,Equalized Odds,Unsupervised Fairness,Supervised Fairness
0,LR_8_features,binary 0/1,age_cat,False,False,True,False,False,False,False,False,True,True,False,False,False,False,False
1,LR_8_features,binary 0/1,race,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False
2,LR_8_features,binary 0/1,sex,False,True,False,True,False,True,True,True,True,False,False,False,True,False,False
